# ML Modeling Notebook
In this notebook, I will be working on different algorithms, No just default values too, I'll tweak the hyperparameters and and see how they affect each other.

List of models that will be tested:
- Logistic Regression
- KNN
- Naive Bayes
- Decision Tree
- Random Forest
- XGBoost
- LightGBM

In [1]:
import pandas as pd

# 1. Read the newly saved CSV file
df = pd.read_csv('data/cleaned_loan_data.csv')

# 2. Extract the column names into a native Python list
clean_columns = df.columns.tolist()

# 3. Print the total count and the list of columns
print(f"Total columns loaded: {len(clean_columns)}\n")
print("--- Cleaned Dataset Columns ---")
for col in clean_columns:
    print(col)

Total columns loaded: 16

--- Cleaned Dataset Columns ---
NAME_INCOME_TYPE
NAME_EDUCATION_TYPE
NAME_FAMILY_STATUS
CNT_CHILDREN
CNT_FAM_MEMBERS
AMT_INCOME_TOTAL
FLAG_OWN_CAR
FLAG_OWN_REALTY
NAME_HOUSING_TYPE
AMT_CREDIT
TARGET
AGE_YEARS
YEARS_EMPLOYED
FLAG_EMP_ANOMALY
CREDIT_INCOME_RATIO
ADULT_IN_HOUSE


In [2]:
# Moved the two FLAG columns here
categorical_cols = [
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 
    'NAME_HOUSING_TYPE', 'NAME_FAMILY_STATUS',
    'FLAG_OWN_CAR', 'FLAG_OWN_REALTY'
]

# Removed the two FLAG columns and the duplicate HOUSING column
numerical_cols = [
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'AMT_INCOME_TOTAL', 
    'AMT_CREDIT', 'AGE_YEARS', 'YEARS_EMPLOYED', 
    'FLAG_EMP_ANOMALY', 'CREDIT_INCOME_RATIO', 'ADULT_IN_HOUSE'
]

# Logistic Regression

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# --- 2. Build the Preprocessor (The Traffic Cop) ---
preprocessor = ColumnTransformer(
    transformers=[
        # Apply standard scaling to numerical features
        ('num', StandardScaler(), numerical_cols),
        
        # Apply One-Hot Encoding to categorical features
        # handle_unknown='ignore' prevents crashes if a weird category pops up in the future
        # drop='first' helps prevent multicollinearity by dropping one binary column per category
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols)
    ])

# --- 3. Build the Full Pipeline ---
# Rebuild the pipeline with the convergence fix and class weights
log_reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        solver='saga',
        l1_ratio=1.0,                
        C=1.0,                       
        max_iter=5000,               # <--- FIX 1: Give the solver more iterations to converge
        class_weight='balanced',     # <--- FIX 2: Force the model to care about defaulters
        random_state=42
    ))
])

# --- 4. Train and Evaluate ---
# Separate features and target
X = df[categorical_cols + numerical_cols]
y = df['TARGET']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Retrain it
print("Training updated pipeline... (This might take a moment with 5000 iterations)")
log_reg_pipeline.fit(X_train, y_train)

# Make predictions
y_pred = log_reg_pipeline.predict(X_test)
y_prob = log_reg_pipeline.predict_proba(X_test)[:, 1]

# --- 5. View Results ---
print("\n--- Pipeline Evaluation ---")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Training updated pipeline... (This might take a moment with 5000 iterations)


C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(



--- Pipeline Evaluation ---
ROC-AUC Score: 0.6371

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.57      0.71     39561
           1       0.12      0.63      0.20      3490

    accuracy                           0.58     43051
   macro avg       0.53      0.60      0.46     43051
weighted avg       0.88      0.58      0.67     43051



## Model Evaluation

In the initial Logistic Regression run, the model achieved an apparent **92% Accuracy**. However, this was a mathematical illusion caused by the severe class imbalance in the dataset. Because the vast majority of applicants naturally repay their loans (Class 0), the algorithm simply guessed "Repaid" 100% of the time, resulting in a **0.00 Recall** for actual defaulters (Class 1). 

To fix this, we introduced `class_weight='balanced'`, forcing the mathematical solver to heavily penalize mistakes made on the rare minority class.

### 📊 Performance Comparison

| Metric | Baseline Model (Unweighted) | Balanced Model (Weighted) | Impact |
| :--- | :--- | :--- | :--- |
| **ROC-AUC Score** | 0.6369 | 0.6371 | Stable. The model's core ability to separate classes remains intact. |
| **Overall Accuracy** | 92% | 58% | **Expected Drop.** The model stopped "lazy guessing." |
| **Defaulter Recall** | 0.00 (Caught 0%) | 0.63 (Caught 63%) | **Massive Improvement.** The model is now actively identifying high-risk applicants. |
| **Defaulter Precision** | 0.00 | 0.12 (12%) | **The Tradeoff.** The model casts a wide net, resulting in a high rate of false alarms. |

### 🔍 Key Takeaways for Credit Risk
1. **Accuracy is a Trap:** In alternative credit scoring, raw accuracy is a dangerous metric. A model that approves everyone will have high accuracy but will bankrupt the lender.
2. **The Recall Victory:** By catching 63% of the actual defaulters, the weighted model protects capital. This is a highly functional starting point.
3. **The Precision-Recall Tradeoff:** The cost of catching those bad loans is a low precision (12%). This means that for every 10 people the model flags as a "Defaulter," 8 or 9 of them were actually safe applicants. 

**Next Steps:** While Logistic Regression provides a solid baseline and excellent interpretability, the low precision suggests that linear decision boundaries are struggling with the complexity of the features. We will move on to the next model.

# K- Nearest Neighbour (KNN)

In [27]:
from sklearn.neighbors import KNeighborsClassifier

# --- Build the KNN Pipeline ---
knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), # Reusing your exact same ColumnTransformer!
    ('classifier', KNeighborsClassifier(
        n_neighbors=5,       # How many neighbors to vote (you can tune this later)
        algorithm='kd_tree', # <--- FIX 1: Use a tree structure instead of brute force
        n_jobs=-1            # <--- FIX 2: Use all CPU cores
    ))
])

print("Training KNN with K-D Tree optimization...")
knn_pipeline.fit(X_train, y_train)

# Make predictions
y_pred_knn = knn_pipeline.predict(X_test)
y_prob_knn = knn_pipeline.predict_proba(X_test)[:, 1]

# Evaluate
print("\n--- KNN Evaluation ---")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_knn):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))

Training KNN with K-D Tree optimization...

--- KNN Evaluation ---
ROC-AUC Score: 0.5411

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.99      0.95     39561
           1       0.13      0.01      0.02      3490

    accuracy                           0.91     43051
   macro avg       0.52      0.50      0.49     43051
weighted avg       0.86      0.91      0.88     43051



## Model Evaluation: K-Nearest Neighbors (KNN)

Our second model, K-Nearest Neighbors (KNN), delivered a **91% Overall Accuracy**, but an incredibly poor **1% Recall** for actual defaulters (Class 1). With an **ROC-AUC score of 0.5411**, this model performs barely better than a random coin flip (0.50). 

This complete failure is highly educational and highlights the structural limitations of distance-based algorithms on large, imbalanced, high-dimensional tabular datasets.

### 📊 Performance Comparison: Logistic Regression vs. KNN

| Metric | Balanced Logistic Regression | KNN (K=5, K-D Tree) | Core Implication |
| :--- | :--- | :--- | :--- |
| **ROC-AUC Score** | **0.6371** | 0.5411 | KNN loses almost all predictive power to separate classes. |
| **Overall Accuracy** | 58% | **91%** | KNN falls hard into the lazy-guessing accuracy trap. |
| **Defaulter Recall (Class 1)** | **0.63 (Caught 63%)** | 0.01 (Caught 1%) | **99% of defaulters escaped** detection under KNN. |
| **Defaulter Precision (Class 1)**| 0.12 (12%) | **0.13 (13%)** | Even when KNN flags a default, it is highly inaccurate. |

### 🔍 Why Did KNN Completely Fail?

1. **The Majority Vote Trap (Class Imbalance)**
KNN makes predictions by finding the $K$ closest data points and taking a majority vote. Because 92% of our dataset consists of safe payers (Class 0), any random point dropped into the multi-dimensional space is overwhelmingly likely to be surrounded by Class 0 neighbors. Without a built-in mechanism to weigh classes differently (like the `class_weight='balanced'` parameter we used in Logistic Regression), the minority class is completely drowned out.

2. **The Curse of Dimensionality**
KNN relies on Euclidean distance to determine similarity. As we engineered new numerical ratios and One-Hot Encoded categorical variables (like `NAME_INCOME_TYPE` and `NAME_HOUSING_TYPE`), we drastically increased the number of columns (dimensions). In high-dimensional spaces, the mathematical volume grows exponentially, causing all data points to become nearly equidistant from one another. When "everyone is a neighbor," the concept of distance loses its meaning, reducing the model to near-random guessing.

**Conclusion:** Distance-based geometric models like KNN are fundamentally unsuited for large credit risk datasets with heavy class imbalances and high dimensionality. 

# Naive Bayes

In [30]:
from sklearn.naive_bayes import GaussianNB

# --- Build the Naive Bayes Pipeline ---
nb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), # Reusing your exact same ColumnTransformer!
    ('classifier', GaussianNB(
        # Override the baseline probabilities to force it to care about defaults
        priors=[0.5, 0.5] 
    ))
])

print("Training Gaussian Naive Bayes...")
nb_pipeline.fit(X_train, y_train)

# Make predictions
y_pred_nb = nb_pipeline.predict(X_test)
y_prob_nb = nb_pipeline.predict_proba(X_test)[:, 1]

# Evaluate
print("\n--- Naive Bayes Evaluation ---")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_nb):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))

Training Gaussian Naive Bayes...

--- Naive Bayes Evaluation ---
ROC-AUC Score: 0.6193

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.62      0.75     39561
           1       0.12      0.57      0.20      3490

    accuracy                           0.62     43051
   macro avg       0.53      0.60      0.47     43051
weighted avg       0.88      0.62      0.70     43051



## Model Evaluation: Gaussian Naive Bayes

Our third model, Gaussian Naive Bayes, approached the credit risk problem using pure probability (Bayes' Theorem) rather than geometric distances or linear boundaries. By overriding the baseline probabilities (`priors=[0.5, 0.5]`), we successfully forced the algorithm to pay attention to the minority class, completely avoiding the lazy-guessing trap that destroyed our KNN model.

### 📊 Performance Comparison: The Leaderboard

| Metric | Logistic Regression (Weighted) | Gaussian Naive Bayes (Adjusted Priors) | KNN (K-D Tree) |
| :--- | :--- | :--- | :--- |
| **ROC-AUC Score** | **0.6371** | 0.6193 | 0.5411 |
| **Defaulter Recall (Class 1)** | **0.63 (63%)** | 0.57 (57%) | 0.01 (1%) |
| **Defaulter Precision (Class 1)**| 0.12 (12%) | 0.12 (12%) | 0.13 (13%) |
| **Overall Accuracy** | 58% | **62%** | 91% (Illusion) |

### 🔍 Why Did Logistic Regression Beat Naive Bayes?

While Naive Bayes successfully caught a respectable 57% of actual defaulters, it was mathematically outperformed by Logistic Regression in the ROC-AUC score (0.619 vs 0.637). This happened because of the **"Naive" Assumption**.

1. **The Flaw of Independence:** Naive Bayes strictly assumes that every single feature is completely independent of the others. In our credit dataset, this is mathematically false. For example, our engineered `CREDIT_INCOME_RATIO` is heavily correlated with `AMT_INCOME_TOTAL`, and `YEARS_EMPLOYED` is correlated with `AGE_YEARS`. 
2. **The Logistic Advantage:** Logistic Regression is capable of understanding these correlations and adjusting the $\beta$ weights accordingly. Naive Bayes gets mathematically confused by correlated features, effectively "double-counting" their importance and lowering its overall ranking power (ROC-AUC).

**Conclusion:** Gaussian Naive Bayes is a massive step up from KNN because of its speed and ability to handle imbalanced data via prior adjustments. However, its core assumption of feature independence prevents it from capturing the complex, correlated realities of financial data. Logistic Regression remains our top performer. 

# DECISION TREES

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# --- 1. Define the Decision Tree Pipeline ---
# Notice we still use class_weight='balanced'
dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), # Reusing your existing ColumnTransformer
    ('classifier', DecisionTreeClassifier(class_weight='balanced', random_state=42))
])

# --- 2. Define the Hyperparameter Grid ---
# The 'classifier__' prefix is required because the model is inside a Pipeline
param_distributions = {
    'classifier__max_depth': [None, 5, 10, 15, 20, 25],
    'classifier__min_samples_split': randint(2, 50),
    'classifier__min_samples_leaf': randint(1, 50),
    'classifier__criterion': ['gini', 'entropy']
}

# --- 3. Set up RandomizedSearchCV ---
# n_iter=20 means it will randomly pick and test 20 different combinations
# cv=3 means it uses 3-fold cross-validation to ensure the score is stable
random_search_dt = RandomizedSearchCV(
    estimator=dt_pipeline,
    param_distributions=param_distributions,
    n_iter=60,                 
    scoring='roc_auc',         # <--- Optimizing for Ranking Power!
    cv=3,                      
    n_jobs=-1,                 # Use all CPU cores
    random_state=42,
    verbose=2                  # Prints progress so you aren't staring at a blank screen
)

print("Starting Randomized Search for Decision Tree...")
random_search_dt.fit(X_train, y_train)

# --- 4. Extract and Evaluate the Best Model ---
print("\n--- Best Hyperparameters Found ---")
print(random_search_dt.best_params_)

best_dt_model = random_search_dt.best_estimator_

# Make predictions using the winning model
y_pred_dt = best_dt_model.predict(X_test)
y_prob_dt = best_dt_model.predict_proba(X_test)[:, 1]

# Evaluate
print("\n--- Optimized Decision Tree Evaluation ---")
print(f"Best Cross-Validation ROC-AUC: {random_search_dt.best_score_:.4f}")
print(f"Test Set ROC-AUC Score: {roc_auc_score(y_test, y_prob_dt):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

Starting Randomized Search for Decision Tree...
Fitting 3 folds for each of 60 candidates, totalling 180 fits

--- Best Hyperparameters Found ---
{'classifier__criterion': 'gini', 'classifier__max_depth': 5, 'classifier__min_samples_leaf': 20, 'classifier__min_samples_split': 29}

--- Optimized Decision Tree Evaluation ---
Best Cross-Validation ROC-AUC: 0.6158
Test Set ROC-AUC Score: 0.6229

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.68      0.79     39561
           1       0.12      0.51      0.20      3490

    accuracy                           0.67     43051
   macro avg       0.53      0.59      0.49     43051
weighted avg       0.87      0.67      0.74     43051



## Model Evaluation: Optimized Decision Tree & The Limits of a Single Flowchart

Our fourth model utilized a Decision Tree optimized via `RandomizedSearchCV`. To combat the algorithm's natural tendency to overfit, the grid search severely restricted the model's complexity, settling on a very shallow `max_depth` of 5.

### 📊 Performance Comparison: The Leaderboard

| Metric | Logistic Regression (Weighted) | Decision Tree (Optimized) | Naive Bayes |
| :--- | :--- | :--- | :--- |
| **ROC-AUC Score** | **0.6371** | 0.6229 | 0.6193 |
| **Defaulter Recall** | **0.63 (63%)** | 0.51 (51%) | 0.57 (57%) |
| **Overall Accuracy** | 58% | **67%** | 62% |

### 🔍 Why Did Logistic Regression Win?
While the Decision Tree is highly interpretable, a single tree is fundamentally a "weak learner." Because it makes rigid, orthogonal splits (90-degree cuts in the data space), it struggles to map smooth, linear relationships between continuous numerical variables like `AMT_INCOME_TOTAL` and `CREDIT_INCOME_RATIO`. To capture those smooth curves, the tree would need to grow very deep, which immediately leads to overfitting. 

**Conclusion:** A single Decision Tree is too rigid to capture the nuance of this dataset without overfitting. However, this perfectly sets the stage for our next algorithm family. We will now take this concept and scale it horizontally using **Ensemble Methods**.

# RANDOM FOREST

In [7]:
from sklearn.ensemble import RandomForestClassifier

# --- 1. Define the Random Forest Pipeline ---
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), 
    ('classifier', RandomForestClassifier(
        class_weight='balanced_subsample', # Recalculates weights for every individual tree
        random_state=42,
        n_jobs=-1                          # Use all CPU cores (critical for building 100+ trees)
    ))
])

# --- 2. Define the Hyperparameter Grid ---
# We cast a wide net to let the randomized search find the optimal balance
param_distributions_rf = {
    'classifier__n_estimators': randint(50, 300),          # How many trees
    'classifier__max_features': ['sqrt', 'log2', None],    # How to randomize the columns
    'classifier__max_depth': [None, 10, 20, 30],           # Let them grow deep!
    'classifier__min_samples_split': randint(2, 20),
    'classifier__min_samples_leaf': randint(1, 10)
}

# --- 3. Set up RandomizedSearchCV ---
random_search_rf = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions_rf,
    n_iter=20,                 # Test 20 different random configurations
    scoring='roc_auc',         # Optimize for Ranking Power
    cv=3,                      # 3-Fold Cross Validation
    n_jobs=-1,                 
    random_state=42,
    verbose=2                  
)

print("Starting Randomized Search for Random Forest... (This will take a few minutes)")
random_search_rf.fit(X_train, y_train)

# --- 4. Extract and Evaluate ---
print("\n--- Best Hyperparameters Found ---")
print(random_search_rf.best_params_)

best_rf_model = random_search_rf.best_estimator_

y_pred_rf = best_rf_model.predict(X_test)
y_prob_rf = best_rf_model.predict_proba(X_test)[:, 1]

print("\n--- Optimized Random Forest Evaluation ---")
print(f"Best Cross-Validation ROC-AUC: {random_search_rf.best_score_:.4f}")
print(f"Test Set ROC-AUC Score: {roc_auc_score(y_test, y_prob_rf):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

Starting Randomized Search for Random Forest... (This will take a few minutes)
Fitting 3 folds for each of 20 candidates, totalling 60 fits

--- Best Hyperparameters Found ---
{'classifier__max_depth': 10, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 9, 'classifier__min_samples_split': 18, 'classifier__n_estimators': 268}

--- Optimized Random Forest Evaluation ---
Best Cross-Validation ROC-AUC: 0.6355
Test Set ROC-AUC Score: 0.6445

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.65      0.77     39561
           1       0.12      0.55      0.20      3490

    accuracy                           0.64     43051
   macro avg       0.53      0.60      0.49     43051
weighted avg       0.88      0.64      0.72     43051



## Model Evaluation: Random Forest 

Our fifth model transitioned from a single, rigid Decision Tree to an ensemble method: the Random Forest. By building 268 independent decision trees and averaging their predictions, the model successfully mapped complex, non-linear relationships without overfitting to the training data.

### 📊 Performance Comparison:

| Metric | Random Forest (Optimized) | Logistic Regression | Decision Tree |
| :--- | :--- | :--- | :--- |
| **ROC-AUC Score** | **0.6445** | 0.6371 | 0.6229 |
| **Defaulter Recall** | 0.55 (55%) | **0.63 (63%)** | 0.51 (51%) |
| **Defaulter Precision**| 0.12 (12%) | 0.12 (12%) | 0.12 (12%) |
| **Overall Accuracy** | 64% | 58% | 67% |

### 🔍 Why Did Random Forest Take the Lead?

The Random Forest achieved the highest ranking power (ROC-AUC) out of all models tested so far. This was driven by two key hyperparameter optimizations discovered during the randomized search:

1. **`max_features='sqrt'`:** Instead of letting every tree look at the dominant features (like `CREDIT_INCOME_RATIO`), the algorithm forced each tree to build its flowchart using a random, restricted subset of columns. This forced the ensemble to discover hidden predictive power in "weaker" features, drastically improving the model's overall intelligence.
2. **`class_weight='balanced_subsample'`:** The model mathematically adjusted the severe 92/8 class imbalance for every single tree based on its specific random data slice, ensuring that the rare defaulters were never ignored.

**Conclusion:** The Random Forest proved that non-linear ensemble methods are superior for this dataset. However, its Recall slightly trailed Logistic Regression at the default 50% threshold. For our next model, we will move to an advanced, sequential ensemble algorithm—**XGBoost**—to see if we can push both the ROC-AUC and the Recall even higher.

# XGBoost

In [5]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# --- 1. Calculate the Imbalance Ratio dynamically ---
# count(Class 0) / count(Class 1)
imbalance_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Calculated scale_pos_weight: {imbalance_ratio:.2f}")

# --- 2. Define the XGBoost Pipeline ---
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), 
    ('classifier', xgb.XGBClassifier(
        scale_pos_weight=imbalance_ratio, # The XGBoost way to handle imbalance
        objective='binary:logistic',      # We want probabilities
        eval_metric='auc',                # Optimize the internal engine for AUC
        random_state=42,
        n_jobs=-1
    ))
])

# --- 3. Define the Hyperparameter Grid ---
param_distributions_xgb = {
    'classifier__n_estimators': randint(100, 300),
    'classifier__learning_rate': uniform(0.01, 0.2), # How aggressively it fixes mistakes
    'classifier__max_depth': randint(3, 8),          # XGBoost prefers shallower trees than RF
    'classifier__subsample': uniform(0.6, 0.4),      # Prevent overfitting
    'classifier__colsample_bytree': uniform(0.6, 0.4) 
}

# --- 4. Set up RandomizedSearchCV ---
random_search_xgb = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions_xgb,
    n_iter=60,                 
    scoring='roc_auc',         
    cv=3,                      
    n_jobs=-1,                 
    random_state=42,
    verbose=2                  
)

print("\nStarting Randomized Search for XGBoost... (Let it cook)")
random_search_xgb.fit(X_train, y_train)

# --- 5. Extract and Evaluate ---
print("\n--- Best Hyperparameters Found ---")
print(random_search_xgb.best_params_)

best_xgb_model = random_search_xgb.best_estimator_

y_pred_xgb = best_xgb_model.predict(X_test)
y_prob_xgb = best_xgb_model.predict_proba(X_test)[:, 1]

print("\n--- Optimized XGBoost Evaluation ---")
print(f"Best Cross-Validation ROC-AUC: {random_search_xgb.best_score_:.4f}")
print(f"Test Set ROC-AUC Score: {roc_auc_score(y_test, y_prob_xgb):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

Calculated scale_pos_weight: 11.34

Starting Randomized Search for XGBoost... (Let it cook)
Fitting 3 folds for each of 60 candidates, totalling 180 fits

--- Best Hyperparameters Found ---
{'classifier__colsample_bytree': np.float64(0.942529716751237), 'classifier__learning_rate': np.float64(0.09090162542443801), 'classifier__max_depth': 3, 'classifier__n_estimators': 251, 'classifier__subsample': np.float64(0.9403713795070051)}

--- Optimized XGBoost Evaluation ---
Best Cross-Validation ROC-AUC: 0.6447
Test Set ROC-AUC Score: 0.6537

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.62      0.75     39561
           1       0.12      0.59      0.20      3490

    accuracy                           0.62     43051
   macro avg       0.53      0.61      0.48     43051
weighted avg       0.88      0.62      0.71     43051



## The Final Showdown: Logistic Regression vs. XGBoost

After evaluating linear, distance-based, probabilistic, and ensemble models, the experiment culminated in a showdown between our optimized baseline (Logistic Regression) and our optimized champion (XGBoost). 

While their classification reports appear similar at a glance, their underlying mathematical performance reveals a clear winner.

### 📊 Performance Comparison: The Top Two

| Metric | Logistic Regression (Weighted) | XGBoost (scale_pos_weight) | Why it Matters |
| :--- | :--- | :--- | :--- |
| **ROC-AUC Score** | 0.6371 | **0.6537** | **The True Metric.** XGBoost is mathematically superior at ranking a defaulter as riskier than a safe payer. |
| **Defaulter Recall (at 50%)**| **0.63 (63%)** | 0.59 (59%) | LR naturally outputs higher baseline probabilities, pushing more predictions over the default 50% line. |
| **Defaulter Precision** | 0.12 (12%) | **0.12 (12%)** | Both models struggle with false positives, an expected tradeoff when aggressively hunting minority classes. |
| **Overall Accuracy** | 58% | **62%** | XGBoost retains a better balance of the majority class without sacrificing predictive power. |

### 🔍 Architectural Differences
1. **The Straight Line vs. The Compounding Flowchart:** Logistic Regression relies on drawing a single, straight mathematical boundary through high-dimensional space. It struggles with non-linear relationships. XGBoost uses hundreds of shallow decision trees built sequentially, with each new tree explicitly designed to fix the errors of the previous one.
2. **Threshold Flexibility:** Because XGBoost achieved a notably higher ROC-AUC score, its underlying probability distributions are cleaner. By manually lowering the decision threshold for XGBoost (e.g., from 50% to 42%), we can easily surpass Logistic Regression's Recall while maintaining a superior Precision rate, maximizing the bank's profitability.

**Final Conclusion:** Extreme Gradient Boosting (XGBoost) is officially the highest-performing model for this credit risk dataset.